# Vector Beam Theory Atlas

Ideal radial/azimuthal vector Bessel beams are reference Jones targets. They are not labelled as current-lab outputs under the current two-SLM same-axis/no-waveplate bench.


In [1]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bessel_twin_core as bt
from vbb_study import setup_study, vbb_vector, vbb_style
from vbb_study.publication import vector as vector_schema

PATHS = setup_study.bootstrap(Path.cwd())
PRESET = "fast"
RUN_ID = PATHS.get("run_id") or None
out_csv = PATHS["csv"] / "vector"
out_fig = PATHS["figures"] / "vector"
compat_csv = PATHS["csv"] / "publication_study"
out_csv.mkdir(parents=True, exist_ok=True)
out_fig.mkdir(parents=True, exist_ok=True)
compat_csv.mkdir(parents=True, exist_ok=True)
vbb_style.apply_style()

cfg = bt.default_config(PRESET)
design = bt.compute_design_from_targets(cfg.laser, cfg.target, cfg.material)
grid = bt.make_xy_grid(256, 0.18 * bt.um)
KR = 0.95 / bt.um
WAIST = 48.0 * bt.um
ELL_VALUES = (1, 3)


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [ ]:
# STAGE88: visible editable controls for exploratory notebook use.
# Edit NOTEBOOK_CONTROLS below and re-run this cell to apply parameter
# overrides to `cfg` before running any study cell below.
from vbb_study.publication import notebook_controls as nb_controls
from vbb_study.config import um as _um

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(
    stage='vector',
    # ── edit these to override the base configuration ───────────────────────
    ell=1,
    target_core_diameter_um=3.0,
    target_bessel_length_um=150.0,
    objective_NA=0.45,
)

# Wire control parameters into `cfg` so downstream cells use them.
_p = NOTEBOOK_CONTROLS.parameters or {}
if "ell" in _p:
    cfg = replace(cfg, target=replace(cfg.target, ell=int(_p["ell"])))
if "target_core_diameter_um" in _p:
    cfg = replace(cfg, target=replace(cfg.target, target_core_diameter_m=float(_p["target_core_diameter_um"]) * _um))
if "target_bessel_length_um" in _p:
    cfg = replace(cfg, target=replace(cfg.target, target_bessel_length_m=float(_p["target_bessel_length_um"]) * _um))
if "objective_NA" in _p:
    cfg = replace(cfg, objective=replace(cfg.objective, NA=float(_p["objective_NA"])))

try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


In [ ]:
# Interactive quicklook — adjust sliders and click "Update plots".
# Runs a fast preview only; nothing is saved.
from vbb_study.publication import notebook_widgets as nbw

_panel = nbw.interactive_quicklook(cfg, method='holographic', preset='fast', ell_range=(0, 6))
display(_panel)


In [2]:
def _stamp(row):
    vector_schema.annotate_vector_row(row, run_id=RUN_ID, qa_status="exploratory")
    return row

def _case_metrics(case, *, target=None):
    radius = vbb_vector.predicted_ring_radius(int(case["ell"]), float(case["kr_m_inv"]))
    roi = vbb_vector.ring_roi(case["grid"], radius, rel_width=0.25)
    stokes = case["stokes"]
    circ = vbb_vector.circularity_residual(stokes, roi)
    row = {
        "ell": int(case["ell"]),
        "ring_radius_um": float(radius / bt.um),
        "petal_count_0deg": vbb_vector.petal_count_and_orientation(case["analyzer"][0], case["grid"], radius)["petal_count"],
        "petal_count_45deg": vbb_vector.petal_count_and_orientation(case["analyzer"][45], case["grid"], radius)["petal_count"],
        "mean_abs_S3_over_S0_roi": circ["mean_abs_s3_over_s0"],
        "max_abs_S3_over_S0_roi": circ["max_abs_s3_over_s0"],
        "total_power_au": float(np.sum(case["total_intensity"]) * float(case["grid"]["dx"]) ** 2),
    }
    if target in {"radial", "azimuthal"}:
        row["orientation_rms_error_rad"] = vbb_vector.mode_orientation_error(
            case["ellipse"]["psi"], case["grid"], roi, target=target, order=max(1, abs(int(case["ell"])))
        )
    return row


In [3]:
rows = []
scalar_rows = []
vector_cases = []

for ell in ELL_VALUES:
    envelope = vbb_vector.scalar_bg_envelope(grid, ell=ell, kr_m_inv=KR, waist_m=WAIST)
    scalar_intensity = np.abs(envelope) ** 2
    scalar_row = _stamp({
        "case_id": f"scalar_reference_ell{ell}",
        "path": "vector_theory_atlas",
        "beam_family": "scalar_reference",
        "model_level": "scalar_reference",
        "generation_method": "scalar_reference",
        "vector_mode": "scalar_reference",
        "vector_model": "scalar_sas_with_jones_overlay",
        "vector_program": "scalar_reference",
        "vector_method": "not_applicable",
        "vector_encoder_hardware": "scalar_holographic_reference",
        "lab_realizable": True,
        "simulation_only": False,
        "requires_element": "none",
        "uses_waveplates": False,
        "uses_two_slm": False,
        "uses_shared_director_axis": False,
        "ell": ell,
        "target_core_diameter_um": float(cfg.target.target_core_diameter_m / bt.um),
        "vortex_main_ring_diameter_um": float(2.0 * vbb_vector.predicted_ring_radius(ell, KR) / bt.um),
        "retained_power_fraction": 1.0,
        "peak_intensity_au": float(np.max(scalar_intensity)),
    })
    scalar_rows.append(scalar_row)
    rows.append(dict(scalar_row))

    for mode in ("radial", "azimuthal"):
        case = vbb_vector.build_analytic_vector_mode(grid, ell=ell, kr_m_inv=KR, waist_m=WAIST, mode=mode)
        vector_cases.append(case)
        row = {
            "case_id": f"ideal_{mode}_ell{ell}",
            "path": "vector_theory_atlas",
            "beam_family": "vector",
            "model_level": "ideal_target",
            "generation_method": "qplate_or_vector_converter",
            "vector_mode": mode,
            "vector_model": "ideal_jones_target",
            "vector_program": "ideal_cylindrical_jones_basis",
            "vector_method": "analytic_reference",
            "vector_encoder_hardware": "not_current_bench",
            "lab_realizable": False,
            "simulation_only": False,
            "requires_element": "qplate_or_vector_mode_converter",
            "uses_waveplates": False,
            "uses_two_slm": False,
            "uses_shared_director_axis": False,
            "ell": ell,
            "target_core_diameter_um": float(cfg.target.target_core_diameter_m / bt.um),
            "vortex_main_ring_diameter_um": float(2.0 * vbb_vector.predicted_ring_radius(ell, KR) / bt.um),
            **_case_metrics(case, target=mode),
        }
        rows.append(_stamp(row))

    rows.append(_stamp({
        "case_id": f"ideal_hybrid_ell{ell}",
        "path": "vector_theory_atlas",
        "beam_family": "vector",
        "model_level": "ideal_target",
        "generation_method": "qplate_or_vector_converter",
        "vector_mode": "hybrid",
        "vector_model": "ideal_jones_target",
        "vector_program": "hybrid_reference_placeholder",
        "vector_method": "analytic_reference",
        "vector_encoder_hardware": "not_current_bench",
        "lab_realizable": False,
        "simulation_only": False,
        "requires_element": "independent_polarisation_axis_modulation",
        "uses_waveplates": False,
        "uses_two_slm": False,
        "uses_shared_director_axis": False,
        "ell": ell,
        "target_core_diameter_um": float(cfg.target.target_core_diameter_m / bt.um),
        "vortex_main_ring_diameter_um": float(2.0 * vbb_vector.predicted_ring_radius(ell, KR) / bt.um),
    }))

atlas = vector_schema.ordered_vector_frame(rows)
atlas.to_csv(out_csv / "vector_beam_theory_atlas.csv", index=False)
pd.DataFrame(scalar_rows).to_csv(out_csv / "vector_atlas_scalar_sas_summary.csv", index=False)
pd.DataFrame([row for row in rows if row["vector_mode"] in {"radial", "azimuthal", "hybrid"}]).to_csv(
    out_csv / "vector_atlas_jones_summary.csv", index=False
)

# Compatibility copies for older publication export paths.
pd.DataFrame(scalar_rows).to_csv(compat_csv / "vector_atlas_scalar_sas_summary.csv", index=False)
pd.DataFrame([row for row in rows if row["vector_mode"] in {"radial", "azimuthal", "hybrid"}]).to_csv(
    compat_csv / "vector_atlas_jones_summary.csv", index=False
)
atlas


,run_id,generated_at_utc,source_schema_version,case_id,preset,path,beam_family,model_level,generation_method,hardware_status,...,vortex_main_ring_diameter_um,retained_power_fraction,peak_intensity_au,ring_radius_um,petal_count_0deg,petal_count_45deg,mean_abs_S3_over_S0_roi,max_abs_S3_over_S0_roi,total_power_au,orientation_rms_error_rad
0,20260604T150311Z,2026-06-04T15:03:56.207841+00:00,1.0.0,scalar_reference_ell1,fast,vector_theory_atlas,scalar_reference,scalar_reference,scalar_reference,current_lab_realizable,...,3.876176,1.0,0.337455,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20260604T150311Z,2026-06-04T15:03:56.313107+00:00,1.0.0,ideal_radial_ell1,fast,vector_theory_atlas,vector,ideal_target,qplate_or_vector_converter,future_hardware_required,...,3.876176,NaN,NaN,1.938088,2.0,2.0,1.464899e-17,4.113807e-17,2.269968e-11,0.914794
2,20260604T150311Z,2026-06-04T15:03:56.425053+00:00,1.0.0,ideal_azimuthal_ell1,fast,vector_theory_atlas,vector,ideal_target,qplate_or_vector_converter,future_hardware_required,...,3.876176,NaN,NaN,1.938088,2.0,2.0,1.464899e-17,4.113807e-17,2.269968e-11,0.914794
3,20260604T150311Z,2026-06-04T15:03:56.425207+00:00,1.0.0,ideal_hybrid_ell1,fast,vector_theory_atlas,vector,ideal_target,qplate_or_vector_converter,future_hardware_required,...,3.876176,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20260604T150311Z,2026-06-04T15:03:56.520784+00:00,1.0.0,scalar_reference_ell3,fast,vector_theory_atlas,scalar_reference,scalar_reference,scalar_reference,current_lab_realizable,...,8.844608,1.0,0.185528,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,20260604T150311Z,2026-06-04T15:03:56.649912+00:00,1.0.0,ideal_radial_ell3,fast,vector_theory_atlas,vector,ideal_target,qplate_or_vector_converter,future_hardware_required,...,8.844608,NaN,NaN,4.422304,6.0,6.0,1.314179e-17,5.267878e-17,2.240345e-11,0.902791
6,20260604T150311Z,2026-06-04T15:03:56.780598+00:00,1.0.0,ideal_azimuthal_ell3,fast,vector_theory_atlas,vector,ideal_target,qplate_or_vector_converter,future_hardware_required,...,8.844608,NaN,NaN,4.422304,6.0,6.0,1.314179e-17,5.267878e-17,2.240345e-11,0.902791
7,20260604T150311Z,2026-06-04T15:03:56.780679+00:00,1.0.0,ideal_hybrid_ell3,fast,vector_theory_atlas,vector,ideal_target,qplate_or_vector_converter,future_hardware_required,...,8.844608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
fig = vbb_vector.plot_analyzer_family_grid(vector_cases[:4])
vbb_style.save_figure(
    fig,
    out_fig / "vector_beam_theory_atlas_analyzer_panels.png",
    "Ideal radial and azimuthal vector Bessel reference targets. These are mathematical Jones targets; under the current same-axis two-SLM bench they are not claimed as current lab generated beams.",
    metadata={"stage": "vector", "figure": "vector_beam_theory_atlas_analyzer_panels"},
)
plt.close(fig)

fig = vbb_vector.plot_polarization_quiver(vector_cases[0], step=12)
vbb_style.save_figure(
    fig,
    out_fig / "vector_beam_theory_atlas_quiver.png",
    "Polarisation quiver for an ideal radial vector target. Hardware status is future_hardware_required under the current bench assumptions.",
    metadata={"stage": "vector", "figure": "vector_beam_theory_atlas_quiver"},
)
plt.close(fig)
